# Walmart Store Sales — TFT raw-pipeline inference

This notebook performs inference only:

```text
W&B Registry → self-contained TFT v7 pipeline → pipeline.predict(raw test.csv) → submission artifact
```

It does not download a checkpoint directly, rebuild a `TimeSeriesDataSet`, merge features, calculate the seasonal fallback, or implement top-series logic. Those responsibilities are embedded in the registered raw-input pipeline created in `model_experiment_TFT.ipynb`.


In [ ]:
%pip install -q "torch>=2.3,<3" "lightning>=2.3,<3" "pytorch-forecasting>=1.2,<2" "wandb>=0.19,<1" "pandas>=2.2,<3" "numpy>=1.26,<3" "matplotlib>=3.8,<4" "kaggle>=1.7,<2" "cloudpickle>=3,<4"


In [ ]:
from __future__ import annotations

import hashlib
import json
import subprocess
from pathlib import Path

import cloudpickle
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import wandb
from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.metrics import MAE


In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print(f"Not running in Colab or Drive unavailable: {exc}")


## Configuration


In [ ]:
CONFIG = {
    "data_dir": "/content/drive/MyDrive/walmart_competition_data",
    "output_dir": "/content/drive/MyDrive/walmart_competition_inference/tft",
    "download_dir": "/content/artifacts/tft_raw_pipeline",
    "wandb_entity": "kende23-n-a",
    "wandb_project": "Walmart-Recruiting---Store-Sales-Forecasting",
    "raw_pipeline_registry_uri": "wandb-registry-model/Walmart_TFT_Raw_Pipeline:champion",
    "raw_pipeline_filename": "walmart_tft_raw_pipeline.pkl",
    "submission_artifact_name": "tft-v7-kaggle-submission",
    "submit_to_kaggle": False,
    "kaggle_competition": "walmart-recruiting-store-sales-forecasting",
    "kaggle_message": "TFT v7 W&B registered raw-input pipeline inference",
}
DATA_DIR = Path(CONFIG["data_dir"])
OUTPUT_DIR = Path(CONFIG["output_dir"])
DOWNLOAD_DIR = Path(CONFIG["download_dir"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
CONFIG


## Load raw test data


In [ ]:
test_raw = pd.read_csv(DATA_DIR / "test.csv", parse_dates=["Date"])
required_test = {"Store", "Dept", "Date", "IsHoliday"}
missing = sorted(required_test.difference(test_raw.columns))
if missing:
    raise ValueError({"missing_raw_test_columns": missing})
test_raw = test_raw.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)
print({"rows": len(test_raw), "columns": test_raw.columns.tolist(), "date_range": (str(test_raw.Date.min().date()), str(test_raw.Date.max().date()))})
display(test_raw.head())


## Download registered TFT raw-input pipeline


In [ ]:
run = wandb.init(
    entity=CONFIG["wandb_entity"],
    project=CONFIG["wandb_project"],
    group="tft-inference",
    job_type="tft_raw_pipeline_inference",
    name="tft_v7_registry_raw_pipeline_inference",
    config=CONFIG,
)
pipeline_artifact = run.use_artifact(CONFIG["raw_pipeline_registry_uri"])
pipeline_dir = Path(pipeline_artifact.download(root=str(DOWNLOAD_DIR)))
pipeline_path = pipeline_dir / CONFIG["raw_pipeline_filename"]
if not pipeline_path.exists():
    candidates = sorted(pipeline_dir.glob("*.pkl"))
    if len(candidates) != 1:
        raise FileNotFoundError({"expected": str(pipeline_path), "candidates": [str(p) for p in candidates]})
    pipeline_path = candidates[0]
with pipeline_path.open("rb") as file:
    pipeline = cloudpickle.load(file)
print({"registry_artifact": pipeline_artifact.name, "pipeline_type": type(pipeline).__name__, "metadata": pipeline.metadata()})


## Predict directly from raw test rows


In [ ]:
predictions = np.asarray(pipeline.predict(test_raw), dtype=np.float64)
if len(predictions) != len(test_raw):
    raise ValueError(f"Pipeline returned {len(predictions)} predictions for {len(test_raw)} raw rows.")
if not np.isfinite(predictions).all() or (predictions < 0).any():
    raise ValueError("Pipeline produced invalid predictions.")
prediction_sha256 = hashlib.sha256(predictions.astype(np.float32).tobytes()).hexdigest()
print({"min": float(predictions.min()), "mean": float(predictions.mean()), "max": float(predictions.max()), "sha256": prediction_sha256})


## Build and log Kaggle submission


In [ ]:
submission = test_raw[["Store", "Dept", "Date"]].copy()
submission["Weekly_Sales"] = predictions
submission.insert(0, "Id", submission["Store"].astype(str) + "_" + submission["Dept"].astype(str) + "_" + submission["Date"].dt.strftime("%Y-%m-%d"))
submission = submission[["Id", "Weekly_Sales"]]
submission_path = OUTPUT_DIR / "tft_v7_submission_registry_raw_pipeline.csv"
submission.to_csv(submission_path, index=False)

manifest = {
    "registry_pipeline_uri": CONFIG["raw_pipeline_registry_uri"],
    "registry_pipeline_artifact": pipeline_artifact.name,
    "pipeline_type": type(pipeline).__name__,
    "pipeline_metadata": pipeline.metadata(),
    "test_rows": int(len(test_raw)),
    "submission_rows": int(len(submission)),
    "prediction_min": float(predictions.min()),
    "prediction_mean": float(predictions.mean()),
    "prediction_max": float(predictions.max()),
    "prediction_sha256": prediction_sha256,
}
manifest_path = OUTPUT_DIR / "tft_raw_pipeline_inference_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2))
fig, ax = plt.subplots(figsize=(8, 4)); ax.hist(predictions, bins=100); ax.set_title("TFT registry raw-pipeline prediction distribution"); ax.set_xlabel("Weekly_Sales"); ax.set_ylabel("count"); plt.tight_layout()
hist_path = OUTPUT_DIR / "tft_raw_pipeline_prediction_histogram.png"; fig.savefig(hist_path, dpi=160); plt.show()

run.log({"inference/submission_rows": len(submission), "inference/prediction_min": manifest["prediction_min"], "inference/prediction_mean": manifest["prediction_mean"], "inference/prediction_max": manifest["prediction_max"], "inference/submission_preview": wandb.Table(dataframe=submission.head(2000)), "inference/prediction_histogram": wandb.Image(str(hist_path))})
artifact = wandb.Artifact(CONFIG["submission_artifact_name"], type="submission", metadata=manifest)
for p in [submission_path, manifest_path, hist_path]: artifact.add_file(str(p))
run.log_artifact(artifact, aliases=["registry-raw-pipeline", "latest"])
run.summary.update(manifest)
print({"submission_path": str(submission_path), "rows": len(submission)})
display(submission.head())


## Optional Kaggle submission


In [ ]:
if CONFIG["submit_to_kaggle"]:
    cmd = ["kaggle", "competitions", "submit", "-c", CONFIG["kaggle_competition"], "-f", str(submission_path), "-m", CONFIG["kaggle_message"]]
    result = subprocess.run(cmd, capture_output=True, text=True, check=False)
    print(result.stdout); print(result.stderr)
    if result.returncode != 0: raise RuntimeError("Kaggle submission failed")
else:
    print("Kaggle submission skipped. Set CONFIG['submit_to_kaggle'] = True to submit.")
run.finish()
